#Ingest circuits.csv file
1. Read the file using Spark DataFrame reader API
2. Add Metadata Columns
    - Source File
    - Ingestion TimeStamp
3. Write to bronze delta table

![image_1781366065933.png](./image_1781366065933.png "image_1781366065933.png")

In [0]:

dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
v_batch_id

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_path = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
source_path

### Step 1- Read the csv file using the DataFrame Reader API

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DoubleType
circuits_schema = StructType([
            StructField('circuitId',StringType()),
            StructField('url',StringType()),
            StructField('circuitName',StringType()),
            StructField('lat',DoubleType()),
            StructField('long',DoubleType()),
            StructField('locality',StringType()),
            StructField('country',StringType())
])

In [0]:
circuits_df = (
    spark.read
         .format("csv")
         .option("header","true")
#        .option("inferSchema","true")
         .schema(circuits_schema)
         .option("mode","FAILFAST")
         .load(source_path)
        )


In [0]:
display(circuits_df)

###Step 2- Add Metadata Columns
- Source File
- Ingestion TimeStamp



In [0]:


circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

In [0]:
v_batch_id

### Step 3- Write to bronze delta table

In [0]:
#circuits_final_df = circuits_final_df.withColumn("batch_id", F.lit(v_batch_id))

In [0]:

"""(
circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .partitionBy('batch_id')
        .option('replaceWhere', f"batch_id = '{v_batch_id}'")
        .saveAsTable(table_name)
)"""

write_to_bronze(
    input_df = circuits_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))